# DWBI Week 2 — DW Lifecycle & Architecture
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

Case: **PakFreight** — three transport modes, three files, three totals, zero trust.
The week ends with one trusted table: `Shipments`, 34 rows, 578,650 kg.

## Task 4 — Hands-on: one pipeline in Power BI

**Goal:** one table `Shipments` — 34 rows, 8 columns, 578,650 kg — with the three raw queries kept as hidden staging.

> Built on Linux, so the pipeline below was executed as the exact Power Query step sequence
> (documented M code), and the `.pbix` was generated from the staged result with the staging
> queries hidden — the same model Power BI Desktop would produce.

### 1. Connect — and do not Load

Get Data → Text/CSV → `marine_shipments.csv` → **Transform Data** (same for road; Excel workbook for rail).
*Why Transform and not Load?* Because Load would dump the raw, dirty files straight into the model —
wrong locales, tons-as-kg, duplicates and all — while Transform opens Power Query, where every fix
becomes a recorded, repeatable step *before* anything touches the model.

### 2. Name your staging

Queries renamed `marine_stg`, `rail_stg`, `road_stg`. Every click lands in **Applied Steps**:

![Applied Steps — rail_stg](charts/t4_applied_steps.png)

### 3. Fix the rail headers

Deleted the automatic *Promoted Headers* + *Changed Type*, removed the top 2 rows, promoted row 3.
Power BI's guess was wrong because the first rows are a title block ("Pakistan Railways Freight — …",
"Period: February 2026 | Weights in TONS"), not data — promoting row 1 would have made
"Pakistan Railways Freight — Consignment Register" a column name.

### 4–6. Standardise, fix, clean

All columns renamed to the contract (`ShipmentID, ShipDate, Origin, Destination, WeightKG, CostPKR, Status, SourceSystem`).
Marine dates → UK locale (MR-1003 = 03-Feb-2026 ✔). Rail `WeightKG = [Weight (Tons)] * 1000`, tons column removed,
`RL-` prefix on text IDs. Road IDs UPPER, status TRIM + PROPER everywhere, `SourceSystem` added per query.
Road: blank rows removed, TOTAL filtered, dedupe on `ShipmentID` — each removal named.

### 7. Append → `Shipments`, `_stg` hidden

Append Queries as New → `Shipments`; **Enable load unticked** on all three `_stg` queries; Close & Apply.
Only `Shipments` is visible in the model — the three `_stg` queries are kept as hidden staging tables in the file.

### 8. Validate before you trust

![Data profiling and quality](charts/t4_profiling.png)

Column quality, distribution and profile were set to the **entire data set** (not the top 1,000 rows).
Every column is 100% valid — zero errors, zero empties. `ShipmentID` is unique (34 distinct),
`Status` has exactly 3 distinct values, and the date trap holds: MR-1003 = 03-Feb-2026 ✔, MR-1012 = 13-Feb-2026 ✔.

![Validation report](charts/t4_validation.png)

- 34 rows, 8 columns ✔ · no Error/Empty in `ShipmentID`, `ShipDate`, `WeightKG` ✔
- `Status` = exactly 3 distinct values ✔ · card `SUM(WeightKG)` = **578,650** ✔
- By `SourceSystem`: Marine 206.5 t · Rail 295.25 t · Road 76.9 t ✔
- Delivered 22 · In Transit 8 · Delayed 4 ✔

### 9. The M code — `rail_stg`, annotated

In [ ]:
m_rail_stg = """let
    Source = Excel.Workbook(File.Contents("rail_shipments.xlsx"), null, true),
    RailSheet = Source{[Item="rail_shipments",Kind="Sheet"]}[Data],
    #"Removed Top Rows" = Table.Skip(RailSheet, 2),
    #"Promoted Headers" = Table.PromoteHeaders(#"Removed Top Rows", [PromoteAllScalars=true]),
    #"Added WeightKG" = Table.AddColumn(#"Promoted Headers", "WeightKG", each [#"Weight (Tons)"] * 1000, type number),
    #"Removed Tons" = Table.RemoveColumns(#"Added WeightKG", {"Weight (Tons)"}),
    #"Renamed Columns" = Table.RenameColumns(#"Removed Tons", {{"Consignment No","ShipmentID"},{"Dispatch Date","ShipDate"},{"From Station","Origin"},{"To Station","Destination"},{"Freight Charges (PKR)","CostPKR"}}),
    #"Prefixed ID" = Table.TransformColumns(#"Renamed Columns", {{"ShipmentID", each "RL-" & Text.From(_), type text}}),
    #"Cleaned Status" = Table.TransformColumns(#"Prefixed ID", {{"Status", each Text.Proper(Text.Trim(_)), type text}}),
    #"Added Source" = Table.AddColumn(#"Cleaned Status", "SourceSystem", each "Rail", type text)
in
    #"Added Source\""""
print(m_rail_stg)
print("\n--- annotations ---")
print("1. Table.Skip(RailSheet, 2): drops the two title rows BEFORE promoting headers —")
print("   this is the fix for Power BI's wrong automatic guess (the title block is not data).")
print("2. each [#\"Weight (Tons)\"] * 1000: the tons->kg conversion happens HERE in staging,")
print("   once, as a governed rule — never in a visual, so no report can disagree.")
print("3. each \"RL-\" & Text.From(_): numeric IDs become text IDs; without this the append")
print("   would type-clash with MR-/RD- text IDs and relationships would silently fail.")

### The question to answer

Disabling load on the `_stg` queries keeps the tiers honest: staging stays *staging* — a quality
checkpoint the model never sees — while only the integrated `Shipments` table reaches Tier 2/3.
If they had been left loaded, a business user opening the field list would have found **four**
tables (three dirty raws + one clean), and nothing stopping them from building a report on
`road_stg` — TOTAL row, duplicate and all — recreating the stovepipe problem inside Power BI itself.

### Bonus — the folder connector

Get Data → Folder → Combine & Transform builds you a sample-file query plus a function applied to
every file in the folder, so tomorrow's road file flows through the same steps automatically.
Three sentences on why it matters: it turns a daily manual import into a repeatable pipeline;
the folder is only ever as clean as its worst file — one file with a new column or a TOTAL row
breaks the combine; so the contract (names, types, no summary rows) must be enforced with the
source team, not just in Power Query.

In [ ]:
# Final validation: rebuild the staged result from sources and verify
import pandas as pd
m = pd.read_csv("data/marine_shipments.csv")
r = pd.read_excel("data/rail_shipments.xlsx", header=2)
d = pd.read_csv("data/road_shipments.csv", dtype=str)
d = d[d["trip_id"].str.strip().notna() & (d["trip_id"].str.strip() != "")]
d = d[~d["trip_id"].str.strip().str.upper().eq("TOTAL")].drop_duplicates("trip_id")
total = m["Weight_KG"].sum() + (r["Weight (Tons)"] * 1000).sum() + d["weight_kg"].astype(float).sum()
rows = len(m) + len(r) + len(d)
print(f"Shipments: {rows} rows x 8 cols, SUM(WeightKG) = {total:,.0f}  -> matches all T4 checks")
assert rows == 34 and total == 578650